In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-proje

In [2]:
# ================================================================
# ARCHITECTURE 4 — HYBRID GQA DECODER TRANSFORMER
# FAST ~195M T4 VERSION
# ================================================================
#
# Decoder-only Transformer
#
# 12 layers
# d_model       = 1024
# Q heads       = 16
# KV heads      = 4
# GQA           = 16Q / 4KV
# RoPE
# SwiGLU
# Pre-LN
# Local causal attention W=256
# Full causal attention every 3rd layer
# Final LayerNorm
# Tied input/output embeddings
#
# No pretrained model
# No pretrained weights
# Pure PyTorch
#
# ------------------------------------------------
# TRAINING
# ------------------------------------------------
#
# Target:
#     16,384 tokens / optimizer step
#
# Preferred:
#     Batch 8
#     Grad accumulation 2
#
# Automatic fallback if B8 OOM:
#     Batch 4
#     Grad accumulation 4
#
# Total:
#     10,000 steps
#     163.84M tokens
#
# ------------------------------------------------
# SPEED
# ------------------------------------------------
#
# Fused QKV
# Fused SwiGLU
# Native GQA
# Fused AdamW when available
# torch.compile(reduce-overhead)
# Activation checkpointing OFF
#
# ------------------------------------------------
# IMPORTANT CUDAGRAPH FIX
# ------------------------------------------------
#
# torch.compiler.cudagraph_mark_step_begin()
# is called once at the start of each logical
# training iteration / optimizer step.
#
# This is important because each optimizer step
# contains multiple compiled forward/backward calls
# due to gradient accumulation.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import math
import json
import time
import random
import gc

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# ================================================================
# 2. CONFIG
# ================================================================

CFG = {

    # ------------------------------------------------------------
    # DATA
    # ------------------------------------------------------------

    "DATA_DIR":
        "/kaggle/input/datasets/achiver07/"
        "phase1-tokens-v40k-clean1",

    "VOCAB_SIZE": 40000,


    # ------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------

    "D_MODEL": 1024,

    "N_LAYERS": 12,

    "N_Q_HEADS": 16,

    "N_KV_HEADS": 4,

    "D_FF": 3328,

    "MAX_SEQ_LEN": 1024,

    "SEQ_LEN": 1024,

    "WINDOW": 256,

    "GLOBAL_EVERY": 3,

    "ROPE_BASE": 10000.0,

    "DROPOUT": 0.0,

    "INIT_STD": 0.02,


    # ------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------

    # Preferred: B8 / accumulation 2.
    # Automatically falls back to B4 / accumulation 4.

    "BATCH_SIZE": 8,

    "GRAD_ACCUM": 2,

    "MAX_STEPS": 10000,

    "LR": 3e-4,

    "MIN_LR_RATIO": 0.10,

    "WARMUP_STEPS": 500,

    "WEIGHT_DECAY": 0.1,

    "GRAD_CLIP": 1.0,


    # ------------------------------------------------------------
    # PERFORMANCE
    # ------------------------------------------------------------

    "USE_FUSED_ADAMW": True,

    "USE_COMPILE": True,

    "COMPILE_MODE": "reduce-overhead",


    # ------------------------------------------------------------
    # VALIDATION
    # ------------------------------------------------------------

    "EVAL_EVERY": 500,

    "EVAL_BATCHES": 30,


    # ------------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------------

    "SAVE_EVERY": 500,

    "OUTPUT_DIR":
        "/kaggle/working/arch4_hybrid_gqa_195m",


    # ------------------------------------------------------------
    # RANDOM SEED
    # ------------------------------------------------------------

    "SEED": 42,
}


# ================================================================
# 3. CLEAN PYTHON STATE ONLY
# ================================================================
# IMPORTANT: this resume script intentionally does NOT call
# torch.cuda.empty_cache() before the first successful CUDA forward.
# A previous device-side assert can poison a CUDA session, and
# empty_cache() will then report the same error.

gc.collect()


# ================================================================
# 4. DEVICE
# ================================================================

DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)


print("Device:", DEVICE)


if DEVICE.type == "cuda":

    props = torch.cuda.get_device_properties(0)

    print(
        "GPU:",
        props.name
    )

    print(
        "GPU memory:",
        f"{props.total_memory / 1024**3:.2f} GB"
    )


# ================================================================
# 5. PERFORMANCE SETTINGS
# ================================================================

torch.set_float32_matmul_precision(
    "high"
)


# ================================================================
# 6. SEED
# ================================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


seed_everything(
    CFG["SEED"]
)


# ================================================================
# 7. PATHS
# ================================================================

DATA_DIR = CFG["DATA_DIR"]


PATHS = {

    "acts_train":
        f"{DATA_DIR}/acts_train.bin",

    "sc_train":
        f"{DATA_DIR}/sc_train.bin",

    "tribunals_train":
        f"{DATA_DIR}/tribunals_train_f.bin",

    "acts_val":
        f"{DATA_DIR}/acts_val.bin",

    "sc_val":
        f"{DATA_DIR}/sc_val.bin",

    "tribunals_val":
        f"{DATA_DIR}/tribunals_val_f.bin",

    "tokenizer":
        f"{DATA_DIR}/tokenizer.json",

    "test":
        f"{DATA_DIR}/test.json",
}


print("\nChecking files...")


for name, path in PATHS.items():

    print(
        f"{name:20s}",
        os.path.exists(path),
        path
    )


os.makedirs(

    CFG["OUTPUT_DIR"],

    exist_ok=True

)


# ================================================================
# 8. TOKENIZER
# ================================================================

print("\nLoading tokenizer...")


with open(

    PATHS["tokenizer"],

    "r",

    encoding="utf-8"

) as f:

    tokenizer_json = json.load(f)


print(
    "Tokenizer loaded."
)


# ================================================================
# 9. BIN DATA
# ================================================================

def load_tokens(path):

    return np.memmap(

        path,

        dtype=np.uint16,

        mode="r",

    )


train_arrays = {

    "acts":
        load_tokens(
            PATHS["acts_train"]
        ),

    "sc":
        load_tokens(
            PATHS["sc_train"]
        ),

    "tribunals":
        load_tokens(
            PATHS["tribunals_train"]
        ),

}


val_arrays = {

    "acts":
        load_tokens(
            PATHS["acts_val"]
        ),

    "sc":
        load_tokens(
            PATHS["sc_val"]
        ),

    "tribunals":
        load_tokens(
            PATHS["tribunals_val"]
        ),

}


print("\nTraining data:")


for name, arr in train_arrays.items():

    print(

        f"{name:12s}: "
        f"{len(arr):,} tokens"

    )


print("\nValidation data:")


for name, arr in val_arrays.items():

    print(

        f"{name:12s}: "
        f"{len(arr):,} tokens"

    )


# ================================================================
# 10. SOURCE MIXTURE
# ================================================================

SOURCE_NAMES = [

    "acts",
    "sc",
    "tribunals",

]


SOURCE_PROBS = np.array(

    [
        0.273,
        0.469,
        0.258,
    ],

    dtype=np.float64,

)


SOURCE_PROBS /= SOURCE_PROBS.sum()


print(

    "\nSource probabilities:",

    dict(
        zip(
            SOURCE_NAMES,
            SOURCE_PROBS
        )
    )

)


# ================================================================
# 11. RANDOM WINDOW SAMPLER
# ================================================================

def sample_batch(

    arrays,
    batch_size,
    seq_len,

):

    xs = np.empty(

        (
            batch_size,
            seq_len,
        ),

        dtype=np.int64,

    )


    ys = np.empty(

        (
            batch_size,
            seq_len,
        ),

        dtype=np.int64,

    )


    source_indices = np.random.choice(

        len(SOURCE_NAMES),

        size=batch_size,

        p=SOURCE_PROBS,

    )


    for b, source_idx in enumerate(

        source_indices

    ):

        source_name = SOURCE_NAMES[

            source_idx

        ]


        data = arrays[

            source_name

        ]


        max_start = (

            len(data)

            -

            seq_len

            -

            1

        )


        start = np.random.randint(

            0,

            max_start + 1,

        )


        xs[b] = data[

            start:

            start + seq_len

        ]


        ys[b] = data[

            start + 1:

            start + seq_len + 1

        ]


    return (

        torch.from_numpy(xs),

        torch.from_numpy(ys),

    )


# ================================================================
# 12. RoPE
# ================================================================

class RotaryEmbedding(nn.Module):

    def __init__(

        self,
        dim,
        max_seq_len,
        base=10000.0,

    ):

        super().__init__()


        assert dim % 2 == 0


        inv_freq = (

            1.0
            /
            (
                base
                **
                (
                    torch.arange(

                        0,

                        dim,

                        2,

                        dtype=torch.float32,

                    )

                    /

                    dim

                )

            )

        )


        self.register_buffer(

            "inv_freq",

            inv_freq,

            persistent=False,

        )


        self._build_cache(

            max_seq_len

        )


    def _build_cache(

        self,
        seq_len,

    ):

        positions = torch.arange(

            seq_len,

            dtype=torch.float32,

            device=self.inv_freq.device,

        )


        freqs = torch.outer(

            positions,

            self.inv_freq,

        )


        self.register_buffer(

            "cos_cached",

            freqs.cos(),

            persistent=False,

        )


        self.register_buffer(

            "sin_cached",

            freqs.sin(),

            persistent=False,

        )


    def forward(

        self,
        seq_len,
        device,

    ):

        if (

            seq_len

            >

            self.cos_cached.shape[0]

        ):

            self._build_cache(

                seq_len

            )


        return (

            self.cos_cached[

                :seq_len

            ].to(device),

            self.sin_cached[

                :seq_len

            ].to(device),

        )


def rotate_half(x):

    half = x.shape[-1] // 2

    x1 = x[..., :half]

    x2 = x[..., half:]

    return torch.cat(

        [-x2, x1],

        dim=-1,

    )


def apply_rope(

    q,
    k,
    cos,
    sin,

):

    cos = torch.cat(

        [
            cos,
            cos,
        ],

        dim=-1,

    )


    sin = torch.cat(

        [
            sin,
            sin,
        ],

        dim=-1,

    )


    cos = cos[
        None,
        None,
        :,
        :
    ]


    sin = sin[
        None,
        None,
        :,
        :
    ]


    q = (

        q * cos

        +

        rotate_half(q) * sin

    )


    k = (

        k * cos

        +

        rotate_half(k) * sin

    )


    return q, k


# ================================================================
# 13. FUSED SWIGLU
# ================================================================

class SwiGLU(nn.Module):

    def __init__(

        self,
        d_model,
        d_ff,

    ):

        super().__init__()


        # One GEMM instead of two.

        self.in_proj = nn.Linear(

            d_model,

            2 * d_ff,

            bias=False,

        )


        self.out_proj = nn.Linear(

            d_ff,

            d_model,

            bias=False,

        )


    def forward(self, x):

        gate, up = self.in_proj(x).chunk(

            2,

            dim=-1,

        )


        return self.out_proj(

            F.silu(gate) * up

        )


# ================================================================
# 14. GQA ATTENTION
# ================================================================

class GQAAttention(nn.Module):

    def __init__(

        self,

        d_model,

        n_q_heads,

        n_kv_heads,

        max_seq_len,

        window,

        is_global,

        rope_base,

        dropout=0.0,

    ):

        super().__init__()


        assert (

            d_model % n_q_heads == 0

        )


        assert (

            n_q_heads % n_kv_heads == 0

        )


        self.d_model = d_model

        self.n_q_heads = n_q_heads

        self.n_kv_heads = n_kv_heads

        self.head_dim = (

            d_model // n_q_heads

        )


        self.window = window

        self.is_global = is_global


        # --------------------------------------------------------
        # FUSED QKV PROJECTION
        #
        # Q = 16 × 64 = 1024
        # K =  4 × 64 = 256
        # V =  4 × 64 = 256
        #
        # Total = 1536
        # --------------------------------------------------------

        q_dim = (

            n_q_heads *

            self.head_dim

        )


        kv_dim = (

            n_kv_heads *

            self.head_dim

        )


        self.qkv_proj = nn.Linear(

            d_model,

            q_dim + kv_dim + kv_dim,

            bias=False,

        )


        self.out_proj = nn.Linear(

            d_model,

            d_model,

            bias=False,

        )


        # --------------------------------------------------------
        # RoPE
        # --------------------------------------------------------

        self.rope = RotaryEmbedding(

            self.head_dim,

            max_seq_len,

            rope_base,

        )


        self.dropout = dropout


        self.kv_repeat = (

            n_q_heads

            //

            n_kv_heads

        )


        # --------------------------------------------------------
        # Local causal mask
        # --------------------------------------------------------

        i = torch.arange(

            max_seq_len

        )[:, None]


        j = torch.arange(

            max_seq_len

        )[None, :]


        local_mask = (

            (j <= i)

            &

            (

                j >=

                i - window + 1

            )

        )


        self.register_buffer(

            "local_mask",

            local_mask,

            persistent=False,

        )


    def forward(self, x):

        B, T, C = x.shape


        # --------------------------------------------------------
        # FUSED QKV
        # --------------------------------------------------------

        qkv = self.qkv_proj(x)


        q_dim = (

            self.n_q_heads *

            self.head_dim

        )


        kv_dim = (

            self.n_kv_heads *

            self.head_dim

        )


        q, k, v = torch.split(

            qkv,

            [

                q_dim,

                kv_dim,

                kv_dim,

            ],

            dim=-1,

        )


        # --------------------------------------------------------
        # Q reshape
        # --------------------------------------------------------

        q = q.view(

            B,

            T,

            self.n_q_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # K reshape
        # --------------------------------------------------------

        k = k.view(

            B,

            T,

            self.n_kv_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # V reshape
        # --------------------------------------------------------

        v = v.view(

            B,

            T,

            self.n_kv_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # RoPE
        # --------------------------------------------------------

        cos, sin = self.rope(

            T,

            x.device,

        )


        q, k = apply_rope(

            q,

            k,

            cos,

            sin,

        )


        # --------------------------------------------------------
        # Native GQA
        #
        # Q = 16 heads
        # K = 4 heads
        # V = 4 heads
        # --------------------------------------------------------

        dropout_p = (

            self.dropout

            if self.training

            else 0.0

        )


        if self.is_global:

            # ----------------------------------------------------
            # FULL CAUSAL ATTENTION
            # ----------------------------------------------------

            y = F.scaled_dot_product_attention(

                q,

                k,

                v,

                attn_mask=None,

                dropout_p=dropout_p,

                is_causal=True,

                enable_gqa=True,

            )

        else:

            # ----------------------------------------------------
            # LOCAL CAUSAL ATTENTION
            # ----------------------------------------------------

            mask = self.local_mask[

                :T,

                :T

            ]


            y = F.scaled_dot_product_attention(

                q,

                k,

                v,

                attn_mask=mask,

                dropout_p=dropout_p,

                is_causal=False,

                enable_gqa=True,

            )


        # --------------------------------------------------------
        # Merge heads
        # --------------------------------------------------------

        y = y.transpose(

            1,

            2,

        ).contiguous()


        y = y.view(

            B,

            T,

            C,

        )


        return self.out_proj(y)


# ================================================================
# 15. TRANSFORMER BLOCK
# ================================================================

class Arch4Block(nn.Module):

    def __init__(

        self,

        cfg,

        layer_idx,

    ):

        super().__init__()


        self.layer_idx = layer_idx


        self.is_global = (

            (layer_idx + 1)

            %

            cfg["GLOBAL_EVERY"]

            == 0

        )


        self.norm1 = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        self.attn = GQAAttention(

            d_model=cfg["D_MODEL"],

            n_q_heads=cfg["N_Q_HEADS"],

            n_kv_heads=cfg["N_KV_HEADS"],

            max_seq_len=cfg["MAX_SEQ_LEN"],

            window=cfg["WINDOW"],

            is_global=self.is_global,

            rope_base=cfg["ROPE_BASE"],

            dropout=cfg["DROPOUT"],

        )


        self.norm2 = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        self.ffn = SwiGLU(

            cfg["D_MODEL"],

            cfg["D_FF"],

        )


    def forward(self, x):

        # --------------------------------------------------------
        # Pre-LN attention
        # --------------------------------------------------------

        x = x + self.attn(

            self.norm1(x)

        )


        # --------------------------------------------------------
        # Pre-LN FFN
        # --------------------------------------------------------

        x = x + self.ffn(

            self.norm2(x)

        )


        return x


# ================================================================
# 16. ARCHITECTURE 4
# ================================================================

class Architecture4(nn.Module):

    def __init__(

        self,

        cfg,

    ):

        super().__init__()


        self.cfg = cfg


        # --------------------------------------------------------
        # Token embedding
        # --------------------------------------------------------

        self.token_embedding = nn.Embedding(

            cfg["VOCAB_SIZE"],

            cfg["D_MODEL"],

        )


        # --------------------------------------------------------
        # Transformer
        # --------------------------------------------------------

        self.layers = nn.ModuleList([

            Arch4Block(

                cfg,

                i,

            )

            for i in range(

                cfg["N_LAYERS"]

            )

        ])


        # --------------------------------------------------------
        # Final normalization
        # --------------------------------------------------------

        self.final_norm = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        # --------------------------------------------------------
        # LM head
        # --------------------------------------------------------

        self.lm_head = nn.Linear(

            cfg["D_MODEL"],

            cfg["VOCAB_SIZE"],

            bias=False,

        )


        # --------------------------------------------------------
        # Tied weights
        # --------------------------------------------------------

        self.lm_head.weight = (

            self.token_embedding.weight

        )


        # --------------------------------------------------------
        # Initialization
        # --------------------------------------------------------

        self._init_weights()


    def _init_weights(self):

        std = self.cfg["INIT_STD"]


        # --------------------------------------------------------
        # Token embedding
        # --------------------------------------------------------

        nn.init.normal_(

            self.token_embedding.weight,

            mean=0.0,

            std=std,

        )


        # --------------------------------------------------------
        # Linear layers
        # --------------------------------------------------------

        for module in self.modules():

            if not isinstance(

                module,

                nn.Linear,

            ):

                continue


            # LM head is tied to embedding.

            if (

                module.weight

                is

                self.token_embedding.weight

            ):

                continue


            nn.init.normal_(

                module.weight,

                mean=0.0,

                std=std,

            )


            if module.bias is not None:

                nn.init.zeros_(

                    module.bias

                )


        # --------------------------------------------------------
        # Residual scaling
        # --------------------------------------------------------

        residual_scale = (

            1.0

            /

            math.sqrt(

                2 *

                self.cfg["N_LAYERS"]

            )

        )


        for block in self.layers:

            block.attn.out_proj.weight.data.mul_(

                residual_scale

            )


            block.ffn.out_proj.weight.data.mul_(

                residual_scale

            )


    def forward(

        self,

        input_ids,

        targets=None,

        return_logits=True,

    ):

        # --------------------------------------------------------
        # Embedding
        #
        # NO sqrt(d_model) scaling.
        # --------------------------------------------------------

        x = self.token_embedding(

            input_ids

        )


        # --------------------------------------------------------
        # Transformer
        # --------------------------------------------------------

        for block in self.layers:

            x = block(x)


        # --------------------------------------------------------
        # Final LN
        # --------------------------------------------------------

        x = self.final_norm(x)


        # --------------------------------------------------------
        # LM head
        # --------------------------------------------------------

        logits = self.lm_head(x)


        # --------------------------------------------------------
        # CE
        # --------------------------------------------------------

        loss = None


        if targets is not None:

            loss = F.cross_entropy(

                logits.reshape(

                    -1,

                    logits.size(-1),

                ),

                targets.reshape(

                    -1,

                ),

            )


        if not return_logits:

            return None, loss


        return logits, loss


# ================================================================
# 17. BUILD MODEL
# ================================================================

print(

    "\nBuilding Architecture 4..."

)


raw_model = Architecture4(

    CFG

).to(DEVICE)


# ================================================================
# 18. PARAMETER COUNT
# ================================================================

total_params = sum(

    p.numel()

    for p in raw_model.parameters()

)


trainable_params = sum(

    p.numel()

    for p in raw_model.parameters()

    if p.requires_grad

)


print(

    f"\nParameters: "
    f"{total_params:,}"

)


print(

    f"Trainable: "
    f"{trainable_params:,}"

)


print(

    f"Parameters (M): "
    f"{total_params / 1e6:.3f}M"

)


# ================================================================
# 19. ATTENTION PATTERN
# ================================================================

print(

    "\nAttention pattern:"

)


for i, block in enumerate(

    raw_model.layers,

    start=1,

):

    if block.is_global:

        print(

            f"Layer {i:2d}: GLOBAL"

        )

    else:

        print(

            f"Layer {i:2d}: "
            f"LOCAL W={CFG['WINDOW']}"

        )


# ================================================================
# 20. OPTIMIZER
# ================================================================

decay_params = []

no_decay_params = []


for name, param in raw_model.named_parameters():

    if not param.requires_grad:

        continue


    if (

        param.ndim >= 2

        and

        "norm" not in name.lower()

    ):

        decay_params.append(param)

    else:

        no_decay_params.append(param)


optimizer_kwargs = {

    "lr":
        CFG["LR"],

    "betas":
        (0.9, 0.95),

    "eps":
        1e-8,

}


use_fused = False


if (

    DEVICE.type == "cuda"

    and

    CFG["USE_FUSED_ADAMW"]

):

    try:

        optimizer = torch.optim.AdamW(

            [

                {

                    "params":
                        decay_params,

                    "weight_decay":
                        CFG["WEIGHT_DECAY"],

                },

                {

                    "params":
                        no_decay_params,

                    "weight_decay":
                        0.0,

                },

            ],

            fused=True,

            **optimizer_kwargs,

        )


        use_fused = True


        print(

            "\nOptimizer: FUSED AdamW"

        )


    except Exception as e:

        print(

            "\nFused AdamW unavailable."

        )

        print(

            "Falling back to foreach AdamW."

        )

        print(

            str(e)[:300]

        )


        optimizer = torch.optim.AdamW(

            [

                {

                    "params":
                        decay_params,

                    "weight_decay":
                        CFG["WEIGHT_DECAY"],

                },

                {

                    "params":
                        no_decay_params,

                    "weight_decay":
                        0.0,

                },

            ],

            foreach=True,

            **optimizer_kwargs,

        )

else:

    optimizer = torch.optim.AdamW(

        [

            {

                "params":
                    decay_params,

                "weight_decay":
                    CFG["WEIGHT_DECAY"],

            },

            {

                "params":
                    no_decay_params,

                "weight_decay":
                    0.0,

            },

        ],

        foreach=True,

        **optimizer_kwargs,

    )


# ================================================================
# 21. AMP
# ================================================================

USE_AMP = (

    DEVICE.type == "cuda"

)


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=USE_AMP,

)


# ================================================================
# 22. LEARNING RATE
# ================================================================

def get_lr(step):

    if step < CFG["WARMUP_STEPS"]:

        return (

            CFG["LR"]

            *

            (step + 1)

            /

            CFG["WARMUP_STEPS"]

        )


    progress = (

        step

        -

        CFG["WARMUP_STEPS"]

    ) / (

        CFG["MAX_STEPS"]

        -

        CFG["WARMUP_STEPS"]

    )


    progress = min(

        max(progress, 0.0),

        1.0,

    )


    cosine = (

        0.5

        *

        (

            1.0

            +

            math.cos(

                math.pi * progress

            )

        )

    )


    min_lr = (

        CFG["LR"]

        *

        CFG["MIN_LR_RATIO"]

    )


    return (

        min_lr

        +

        (

            CFG["LR"]

            -

            min_lr

        )

        *

        cosine

    )


def set_lr(

    optimizer,

    lr,

):

    for group in optimizer.param_groups:

        group["lr"] = lr


# ================================================================
# 23. CUDA GRAPH ITERATION MARKER
# ================================================================

def mark_cudagraph_iteration():

    """

    Mark the beginning of one logical iteration when using
    torch.compile(reduce-overhead).

    We intentionally call this ONCE per optimizer step,
    before gradient accumulation begins.

    """

    if not compile_enabled:

        return


    if not hasattr(

        torch,

        "compiler"

    ):

        return


    if not hasattr(

        torch.compiler,

        "cudagraph_mark_step_begin"

    ):

        return


    torch.compiler.cudagraph_mark_step_begin()


# ================================================================
# 24. MEMORY REPORT
# ================================================================

def memory_report():

    if DEVICE.type != "cuda":

        return


    allocated = (

        torch.cuda.memory_allocated()

        /

        1024**3

    )


    reserved = (

        torch.cuda.memory_reserved()

        /

        1024**3

    )


    peak = (

        torch.cuda.max_memory_allocated()

        /

        1024**3

    )


    print(

        f"GPU memory | "

        f"allocated={allocated:.2f} GB | "

        f"reserved={reserved:.2f} GB | "

        f"peak={peak:.2f} GB"

    )



# ================================================================
# 25. RESUME SETTINGS
# ================================================================

# IMPORTANT:
# Run this notebook AFTER using Kaggle -> Restart Session.
# Do not run this in the CUDA-poisoned kernel that produced the
# "device-side assert triggered" error.

RESUME_CHECKPOINT = (
    "/kaggle/input/notebooks/achiver07/"
    "architecture-4-hybrid-local-global-grouped-query/"
    "arch4_hybrid_gqa_195m/best.pt"
)

# New output directory. No checkpoint_XXXXX.pt files are created.
RESUME_OUTPUT_DIR = (
    "/kaggle/working/arch4_hybrid_gqa_195m_resume"
)

os.makedirs(
    RESUME_OUTPUT_DIR,
    exist_ok=True,
)

# The original T4 run fell back to B4 / accumulation 4.
# Keep the same effective 16,384 tokens per optimizer step and do not
# retest B8, which only wastes time and creates avoidable memory churn.
CFG["BATCH_SIZE"] = 4
CFG["GRAD_ACCUM"] = 4
CFG["OUTPUT_DIR"] = RESUME_OUTPUT_DIR

# Optional debug switch. Leave False for normal training speed.
STRICT_BATCH_TOKEN_CHECK = False


# ================================================================
# 26. DEVICE / CUDA DIAGNOSTIC
# ================================================================

print("\n" + "=" * 70)
print("ARCHITECTURE 4 — EXACT CHECKPOINT RESUME")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA version:",
        torch.version.cuda,
    )
    print(
        "PyTorch version:",
        torch.__version__,
    )


# ================================================================
# 27. EXACT DATA LENGTH / TOKEN RANGE CHECK
# ================================================================

EXPECTED_LENGTHS = {
    "acts": 145_554_561,
    "sc": 249_782_556,
    "tribunals": 137_567_074,
}

EXPECTED_VAL_LENGTHS = {
    "acts": 3_368_151,
    "sc": 4_799_399,
    "tribunals": 28_404_707,
}


def check_memmap_lengths():
    for name, expected in EXPECTED_LENGTHS.items():
        actual = len(train_arrays[name])
        if actual != expected:
            raise RuntimeError(
                f"TRAIN LENGTH MISMATCH for {name}: "
                f"got {actual:,}, expected {expected:,}. "
                "The .bin files MUST be loaded as uint16."
            )

    for name, expected in EXPECTED_VAL_LENGTHS.items():
        actual = len(val_arrays[name])
        if actual != expected:
            raise RuntimeError(
                f"VAL LENGTH MISMATCH for {name}: "
                f"got {actual:,}, expected {expected:,}. "
                "The .bin files MUST be loaded as uint16."
            )



def check_token_range_sample(arrays, label):
    """
    Check several deterministic slices instead of scanning every token.
    This catches the previous uint32-loading mistake, where two uint16
    tokens could be packed into one invalid embedding index.
    """
    sample_span = 65_536

    for name, arr in arrays.items():
        n = len(arr)
        starts = [
            0,
            max(0, n - sample_span),
            n // 3,
            (2 * n) // 3,
        ]

        # Add a few deterministic pseudo-random positions.
        rng = np.random.default_rng(
            CFG["SEED"] + len(name)
        )
        if n > sample_span:
            starts.extend(
                rng.integers(
                    0,
                    n - sample_span,
                    size=4,
                ).tolist()
            )

        lo = None
        hi = None

        for start in starts:
            chunk = np.asarray(
                arr[start:start + sample_span]
            )
            if chunk.size == 0:
                continue

            cmin = int(chunk.min())
            cmax = int(chunk.max())
            lo = cmin if lo is None else min(lo, cmin)
            hi = cmax if hi is None else max(hi, cmax)

        if lo is None or hi is None:
            raise RuntimeError(
                f"No token samples available for {label}/{name}."
            )

        if lo < 0 or hi >= CFG["VOCAB_SIZE"]:
            raise RuntimeError(
                f"INVALID TOKEN ID in {label}/{name}: "
                f"range sample [{lo}, {hi}], expected [0, "
                f"{CFG['VOCAB_SIZE'] - 1}]. "
                "Do NOT use uint32/int32 memmap loading."
            )

        print(
            f"{label:10s} {name:10s} "
            f"length={len(arr):,} "
            f"sample_id_range=[{lo}, {hi}]"
        )


check_memmap_lengths()

print("\nVerified exact token-stream lengths:")
for name in SOURCE_NAMES:
    print(
        f"  {name:10s}: {len(train_arrays[name]):,}"
    )

print("\nChecking sampled token-id ranges...")
check_token_range_sample(
    train_arrays,
    "train",
)
check_token_range_sample(
    val_arrays,
    "val",
)


# ================================================================
# 28. LOAD EXACT ORIGINAL CHECKPOINT
# ================================================================

if not os.path.exists(RESUME_CHECKPOINT):
    raise FileNotFoundError(
        "Original checkpoint not found:\n"
        f"{RESUME_CHECKPOINT}"
    )

print("\n" + "=" * 70)
print("LOADING ORIGINAL BEST CHECKPOINT")
print("=" * 70)

checkpoint_size_gb = (
    os.path.getsize(RESUME_CHECKPOINT) / 1024**3
)

print(
    f"Checkpoint size: {checkpoint_size_gb:.3f} GB"
)
print(
    f"Path:\n{RESUME_CHECKPOINT}"
)

# Keep the checkpoint on CPU while loading. This prevents a large
# transient GPU allocation during deserialization.
checkpoint = torch.load(
    RESUME_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

required_keys = {
    "step",
    "model",
    "optimizer",
    "scaler",
    "best_val",
    "cfg",
}
missing_checkpoint_keys = required_keys - set(checkpoint.keys())
if missing_checkpoint_keys:
    raise RuntimeError(
        "Checkpoint is missing keys: "
        f"{sorted(missing_checkpoint_keys)}"
    )

checkpoint_step = int(checkpoint["step"])
checkpoint_best_val = float(checkpoint["best_val"])

print(
    f"Checkpoint step: {checkpoint_step}"
)
print(
    f"Checkpoint best validation: "
    f"{checkpoint_best_val:.10f}"
)

if checkpoint_step >= CFG["MAX_STEPS"]:
    raise RuntimeError(
        f"Checkpoint is already at step {checkpoint_step}; "
        f"target is {CFG['MAX_STEPS']}."
    )


# ---------------------------------------------------------------
# EXACT MODEL STATE
# ---------------------------------------------------------------

checkpoint_model = checkpoint["model"]

model_keys = set(raw_model.state_dict().keys())
checkpoint_model_keys = set(checkpoint_model.keys())

missing_model_keys = model_keys - checkpoint_model_keys
unexpected_model_keys = checkpoint_model_keys - model_keys

if missing_model_keys or unexpected_model_keys:
    raise RuntimeError(
        "Exact Architecture 4 checkpoint/model key mismatch.\n"
        f"Missing keys ({len(missing_model_keys)}): "
        f"{sorted(missing_model_keys)[:10]}\n"
        f"Unexpected keys ({len(unexpected_model_keys)}): "
        f"{sorted(unexpected_model_keys)[:10]}\n"
        "This must be zero/zero; do not use a reconstructed model class."
    )

raw_model.load_state_dict(
    checkpoint_model,
    strict=True,
)

print(
    "Model state loaded STRICTLY: "
    f"{len(model_keys)} keys, 0 missing, 0 unexpected."
)


# ---------------------------------------------------------------
# EXACT OPTIMIZER STATE
# ---------------------------------------------------------------

checkpoint_optimizer = checkpoint["optimizer"]

if len(optimizer.param_groups) != 2:
    raise RuntimeError(
        f"Expected exactly 2 optimizer parameter groups, got "
        f"{len(optimizer.param_groups)}."
    )

if (
    "param_groups" not in checkpoint_optimizer
    or len(checkpoint_optimizer["param_groups"]) != 2
):
    raise RuntimeError(
        "Original checkpoint optimizer does not contain exactly "
        "2 parameter groups."
    )

optimizer.load_state_dict(
    checkpoint_optimizer
)

print(
    "Optimizer state loaded: 2 parameter groups."
)


# ---------------------------------------------------------------
# EXACT AMP SCALER STATE
# ---------------------------------------------------------------

scaler.load_state_dict(
    checkpoint["scaler"]
)

print("AMP scaler state loaded.")

# Preserve the checkpoint's historical best value.
best_val = checkpoint_best_val
start_step = checkpoint_step

# Match the LR implied by the original schedule for the last completed step.
set_lr(
    optimizer,
    get_lr(max(start_step - 1, 0)),
)

# The large CPU checkpoint object is no longer needed.
del checkpoint

del checkpoint_model

del checkpoint_optimizer

gc.collect()


# ================================================================
# 29. RESUME SANITY FORWARD — EAGER, BEFORE COMPILE
# ================================================================

print("\n" + "=" * 70)
print("RESUME SANITY CHECK — EAGER")
print("=" * 70)

compile_enabled = False
raw_model.eval()

x, y = sample_batch(
    train_arrays,
    1,
    CFG["SEQ_LEN"],
)

if (
    int(x.min()) < 0
    or int(x.max()) >= CFG["VOCAB_SIZE"]
    or int(y.min()) < 0
    or int(y.max()) >= CFG["VOCAB_SIZE"]
):
    raise RuntimeError(
        "Sanity batch contains an invalid token id before CUDA embedding."
    )

x = x.to(
    DEVICE,
    non_blocking=True,
)
y = y.to(
    DEVICE,
    non_blocking=True,
)

with torch.no_grad():
    with torch.amp.autocast(
        "cuda",
        enabled=USE_AMP,
        dtype=torch.float16,
    ):
        logits, sanity_loss = raw_model(
            x,
            y,
        )

print(
    f"Loaded-model sanity loss: {sanity_loss.item():.4f}"
)
print(
    f"Expected vicinity: trained model, not random-init loss "
    f"(~{math.log(CFG['VOCAB_SIZE']):.4f})."
)

# Force synchronization here. If the checkpoint or input IDs are invalid,
# fail NOW, before torch.compile is involved.
if DEVICE.type == "cuda":
    torch.cuda.synchronize()

# Extra runtime guard on output shape.
assert logits.shape == (
    1,
    CFG["SEQ_LEN"],
    CFG["VOCAB_SIZE"],
), (
    "Unexpected sanity logits shape: "
    f"{tuple(logits.shape)}"
)

print(
    "Eager sanity forward passed."
)

del logits
del sanity_loss
del x
del y

gc.collect()


# ================================================================
# 30. TORCH.COMPILE
# ================================================================

model = raw_model

if (
    CFG["USE_COMPILE"]
    and hasattr(torch, "compile")
    and DEVICE.type == "cuda"
):
    print("\n" + "=" * 70)
    print("COMPILING MODEL")
    print("=" * 70)

    try:
        model = torch.compile(
            raw_model,
            mode=CFG["COMPILE_MODE"],
            dynamic=False,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        raw_model.train()

        x, y = sample_batch(
            train_arrays,
            CFG["BATCH_SIZE"],
            CFG["SEQ_LEN"],
        )

        if STRICT_BATCH_TOKEN_CHECK:
            if (
                int(x.min()) < 0
                or int(x.max()) >= CFG["VOCAB_SIZE"]
                or int(y.min()) < 0
                or int(y.max()) >= CFG["VOCAB_SIZE"]
            ):
                raise RuntimeError(
                    "Compile warmup batch contains invalid token IDs."
                )

        x = x.to(
            DEVICE,
            non_blocking=True,
        )
        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        # One CUDAGraph logical iteration begins here.
        mark_cudagraph_iteration()

        with torch.amp.autocast(
            "cuda",
            enabled=USE_AMP,
            dtype=torch.float16,
        ):
            logits, loss = model(
                x,
                y,
            )

        scaler.scale(loss).backward()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        optimizer.zero_grad(
            set_to_none=True,
        )

        del logits
        del loss
        del x
        del y

        gc.collect()

        compile_enabled = True
        print("torch.compile: ENABLED")

    except Exception as e:
        print("\ntorch.compile warmup FAILED.")
        print("Falling back to eager PyTorch.")
        print("Reason:", str(e)[:1500])

        optimizer.zero_grad(
            set_to_none=True,
        )

        model = raw_model
        compile_enabled = False
        gc.collect()
else:
    print("\ntorch.compile: DISABLED")


# ================================================================
# 31. TRAINING CONFIG
# ================================================================

tokens_per_microbatch = (
    CFG["BATCH_SIZE"] * CFG["SEQ_LEN"]
)

tokens_per_step = (
    tokens_per_microbatch * CFG["GRAD_ACCUM"]
)

total_training_tokens = (
    tokens_per_step * CFG["MAX_STEPS"]
)

assert tokens_per_step == 16_384

remaining_steps = (
    CFG["MAX_STEPS"] - start_step
)
remaining_tokens = (
    remaining_steps * tokens_per_step
)

print("\n" + "=" * 70)
print("ARCHITECTURE 4 — RESUME CONFIG")
print("=" * 70)
print(f"Parameters: {total_params:,}")
print(f"Parameters (M): {total_params / 1e6:.3f}")
print(f"Resume step: {start_step}")
print(f"Target step: {CFG['MAX_STEPS']}")
print(f"Remaining steps: {remaining_steps}")
print(f"Remaining tokens: {remaining_tokens:,}")
print(f"Batch: {CFG['BATCH_SIZE']}")
print(f"Gradient accumulation: {CFG['GRAD_ACCUM']}")
print(f"Sequence length: {CFG['SEQ_LEN']}")
print(f"Tokens / microbatch: {tokens_per_microbatch:,}")
print(f"Tokens / optimizer step: {tokens_per_step:,}")
print(f"Fused AdamW: {use_fused}")
print(f"torch.compile: {compile_enabled}")
print(f"Best validation so far: {best_val:.4f}")
print(f"Output directory: {RESUME_OUTPUT_DIR}")


# ================================================================
# 32. ROBUST SPLIT CHECKPOINT WRITING
# ================================================================

def atomic_torch_save(obj, path):
    """
    Write to a temporary file and atomically replace the previous file.
    Split files keep every artifact below the ~2 GB combined-checkpoint
    boundary that caused the original iostream failure.
    """
    tmp_path = path + ".tmp"

    try:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        torch.save(
            obj,
            tmp_path,
        )

        os.replace(
            tmp_path,
            path,
        )

    except Exception:
        # Never destroy the previous valid checkpoint on a failed save.
        try:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
        except Exception:
            pass
        raise



def model_state_for_save():
    # State dict is built from the original uncompiled model so key names
    # remain exactly compatible with the original Architecture4 class.
    return raw_model.state_dict()



def save_best_checkpoint(step, val):
    """Save only inference-useful BEST artifacts."""
    best_model_path = os.path.join(
        RESUME_OUTPUT_DIR,
        "best_model.pt",
    )
    best_meta_path = os.path.join(
        RESUME_OUTPUT_DIR,
        "best_meta.json",
    )

    atomic_torch_save(
        model_state_for_save(),
        best_model_path,
    )

    meta = {
        "step": int(step),
        "best_val": float(val),
        "attention": "12-layer hybrid GQA; local W=256; global every 3rd layer",
        "d_model": CFG["D_MODEL"],
        "n_layers": CFG["N_LAYERS"],
        "n_q_heads": CFG["N_Q_HEADS"],
        "n_kv_heads": CFG["N_KV_HEADS"],
        "seq_len": CFG["SEQ_LEN"],
        "batch_size": CFG["BATCH_SIZE"],
        "grad_accum": CFG["GRAD_ACCUM"],
        "tokens_per_optimizer_step": tokens_per_step,
    }

    tmp_meta = best_meta_path + ".tmp"
    with open(
        tmp_meta,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            meta,
            f,
            indent=2,
        )
    os.replace(
        tmp_meta,
        best_meta_path,
    )

    print(
        f"NEW BEST SAVED:\n  {best_model_path}\n  {best_meta_path}"
    )



def save_last_checkpoint(step, best_val):
    """Save LAST model + optimizer separately for exact resume."""
    last_model_path = os.path.join(
        RESUME_OUTPUT_DIR,
        "last_model.pt",
    )
    last_optimizer_path = os.path.join(
        RESUME_OUTPUT_DIR,
        "last_optimizer.pt",
    )
    last_meta_path = os.path.join(
        RESUME_OUTPUT_DIR,
        "last_meta.json",
    )

    # Model file is ~0.8 GB; optimizer file is ~1.5 GB. The original
    # combined file was ~2.18 GB and failed during torch.save().
    atomic_torch_save(
        model_state_for_save(),
        last_model_path,
    )

    atomic_torch_save(
        optimizer.state_dict(),
        last_optimizer_path,
    )

    meta = {
        "step": int(step),
        "best_val": float(best_val),
        "scaler": scaler.state_dict(),
        "cfg": CFG,
        "tokens_per_optimizer_step": int(tokens_per_step),
    }

    tmp_meta = last_meta_path + ".tmp"
    with open(
        tmp_meta,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            meta,
            f,
            indent=2,
        )
    os.replace(
        tmp_meta,
        last_meta_path,
    )

    print(
        f"LAST SAVED at step {step}:\n"
        f"  {last_model_path}\n"
        f"  {last_optimizer_path}\n"
        f"  {last_meta_path}"
    )


# Seed the new BEST artifact with the already-known global best at step 4000.
# This ensures best_model.pt exists even if no later checkpoint improves it.
save_best_checkpoint(
    start_step,
    best_val,
)


# ================================================================
# 33. VALIDATION
# ================================================================

@torch.no_grad()
def evaluate_validation(n_batches=30):
    model.eval()

    source_losses = {
        "acts": [],
        "sc": [],
        "tribunals": [],
    }

    batches_per_source = max(
        1,
        n_batches // len(SOURCE_NAMES),
    )

    for source_name in SOURCE_NAMES:
        data = val_arrays[source_name]

        for _ in range(batches_per_source):
            batch_size = CFG["BATCH_SIZE"]

            xs = np.empty(
                (batch_size, CFG["SEQ_LEN"]),
                dtype=np.int64,
            )
            ys = np.empty(
                (batch_size, CFG["SEQ_LEN"]),
                dtype=np.int64,
            )

            max_start = (
                len(data) - CFG["SEQ_LEN"] - 1
            )

            starts = np.random.randint(
                0,
                max_start + 1,
                size=batch_size,
            )

            for b, start in enumerate(starts):
                xs[b] = data[
                    start:start + CFG["SEQ_LEN"]
                ]
                ys[b] = data[
                    start + 1:start + CFG["SEQ_LEN"] + 1
                ]

            if STRICT_BATCH_TOKEN_CHECK:
                if (
                    int(xs.min()) < 0
                    or int(xs.max()) >= CFG["VOCAB_SIZE"]
                    or int(ys.min()) < 0
                    or int(ys.max()) >= CFG["VOCAB_SIZE"]
                ):
                    raise RuntimeError(
                        f"Invalid validation token ids in {source_name}."
                    )

            x = torch.from_numpy(xs).to(
                DEVICE,
                non_blocking=True,
            )
            y = torch.from_numpy(ys).to(
                DEVICE,
                non_blocking=True,
            )

            if compile_enabled:
                mark_cudagraph_iteration()

            with torch.amp.autocast(
                "cuda",
                enabled=USE_AMP,
                dtype=torch.float16,
            ):
                _, loss = model(
                    x,
                    y,
                    return_logits=False,
                )

            source_losses[source_name].append(
                loss.item()
            )

            del x
            del y
            del loss

    model.train()

    result = {}
    for source_name in SOURCE_NAMES:
        result[source_name] = float(
            np.mean(source_losses[source_name])
        )

    result["mean"] = (
        SOURCE_PROBS[0] * result["acts"]
        + SOURCE_PROBS[1] * result["sc"]
        + SOURCE_PROBS[2] * result["tribunals"]
    )

    return result


# ================================================================
# 34. SPEED WARMUP
# ================================================================

print("\n" + "=" * 70)
print("TRAINING SPEED WARMUP")
print("=" * 70)

model.train()

warmup_steps = 2
optimizer.zero_grad(set_to_none=True)

warmup_start = time.time()

for _ in range(warmup_steps):
    optimizer.zero_grad(set_to_none=True)

    mark_cudagraph_iteration()

    for _ in range(CFG["GRAD_ACCUM"]):
        x, y = sample_batch(
            train_arrays,
            CFG["BATCH_SIZE"],
            CFG["SEQ_LEN"],
        )

        if STRICT_BATCH_TOKEN_CHECK:
            if (
                int(x.min()) < 0
                or int(x.max()) >= CFG["VOCAB_SIZE"]
                or int(y.min()) < 0
                or int(y.max()) >= CFG["VOCAB_SIZE"]
            ):
                raise RuntimeError(
                    "Invalid token id in training warmup batch."
                )

        x = x.to(
            DEVICE,
            non_blocking=True,
        )
        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            "cuda",
            enabled=USE_AMP,
            dtype=torch.float16,
        ):
            logits, loss = model(x, y)
            scaled_loss = (
                loss / CFG["GRAD_ACCUM"]
            )

        scaler.scale(
            scaled_loss
        ).backward()

        del logits
        del loss
        del scaled_loss
        del x
        del y

    # Warmup is not an optimizer step.
    optimizer.zero_grad(set_to_none=True)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

warmup_elapsed = time.time() - warmup_start
warmup_tokens = warmup_steps * tokens_per_step
warmup_tok_sec = (
    warmup_tokens / max(warmup_elapsed, 1e-6)
)

print(
    f"Warmup throughput: {warmup_tok_sec:,.0f} tok/s"
)
memory_report()


# ================================================================
# 35. TRAINING LOOP — RESUME AT STEP 4001
# ================================================================

optimizer.zero_grad(set_to_none=True)
model.train()

start_time = time.time()
last_save_time = start_time

print("\n" + "=" * 70)
print("STARTING RESUMED ARCHITECTURE 4 TRAINING")
print("=" * 70)
print(
    f"Starting from step {start_step}; "
    f"continuing through step {CFG['MAX_STEPS']}."
)

for step in range(
    start_step + 1,
    CFG["MAX_STEPS"] + 1,
):
    # Exactly once per logical optimizer iteration.
    mark_cudagraph_iteration()

    optimizer.zero_grad(
        set_to_none=True,
    )

    total_loss = 0.0

    for _ in range(CFG["GRAD_ACCUM"]):
        x, y = sample_batch(
            train_arrays,
            CFG["BATCH_SIZE"],
            CFG["SEQ_LEN"],
        )

        if STRICT_BATCH_TOKEN_CHECK:
            if (
                int(x.min()) < 0
                or int(x.max()) >= CFG["VOCAB_SIZE"]
                or int(y.min()) < 0
                or int(y.max()) >= CFG["VOCAB_SIZE"]
            ):
                raise RuntimeError(
                    f"Invalid training token id at step {step}."
                )

        x = x.to(
            DEVICE,
            non_blocking=True,
        )
        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            "cuda",
            enabled=USE_AMP,
            dtype=torch.float16,
        ):
            logits, loss = model(
                x,
                y,
            )
            scaled_loss = (
                loss / CFG["GRAD_ACCUM"]
            )

        scaler.scale(
            scaled_loss
        ).backward()

        total_loss += loss.item()

        del logits
        del loss
        del scaled_loss
        del x
        del y

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        raw_model.parameters(),
        CFG["GRAD_CLIP"],
    )

    lr = get_lr(step - 1)
    set_lr(optimizer, lr)

    scaler.step(optimizer)
    scaler.update()

    if step == start_step + 1 or step % 50 == 0:
        elapsed = time.time() - start_time
        completed_steps = step - start_step
        effective_tok_sec = (
            completed_steps * tokens_per_step
            / max(elapsed, 1e-6)
        )

        avg_loss = (
            total_loss / CFG["GRAD_ACCUM"]
        )

        print(
            f"STEP {step:5d}/{CFG['MAX_STEPS']} | "
            f"loss {avg_loss:.4f} | "
            f"lr {lr:.3e} | "
            f"grad {float(grad_norm):.3f} | "
            f"{effective_tok_sec:,.0f} tok/s"
        )

    # ------------------------------------------------------------
    # Validation + BEST update
    # ------------------------------------------------------------
    if step % CFG["EVAL_EVERY"] == 0:
        val = evaluate_validation(
            CFG["EVAL_BATCHES"]
        )

        print(
            "\n"
            f"STEP {step} VALIDATION\n"
            f"  weighted mean = {val['mean']:.4f}\n"
            f"  acts          = {val['acts']:.4f}\n"
            f"  sc            = {val['sc']:.4f}\n"
            f"  tribunals     = {val['tribunals']:.4f}\n"
        )

        memory_report()

        if val["mean"] < best_val:
            best_val = float(val["mean"])
            save_best_checkpoint(
                step,
                best_val,
            )

        # LAST is overwritten in place. No checkpoint_XXXXX.pt files.
        save_last_checkpoint(
            step,
            best_val,
        )
        last_save_time = time.time()

# ================================================================
# 36. FINAL ARTIFACTS
# ================================================================

final_step = CFG["MAX_STEPS"]

# LAST must represent the true final training state even when the target
# step is not an EVAL_EVERY multiple.
save_last_checkpoint(
    final_step,
    best_val,
)

final_model_path = os.path.join(
    RESUME_OUTPUT_DIR,
    "final_model.pt",
)
final_meta_path = os.path.join(
    RESUME_OUTPUT_DIR,
    "final_meta.json",
)

atomic_torch_save(
    model_state_for_save(),
    final_model_path,
)

final_meta = {
    "step": int(final_step),
    "best_val": float(best_val),
    "cfg": CFG,
    "tokens_per_optimizer_step": int(tokens_per_step),
}

_tmp_final_meta = final_meta_path + ".tmp"
with open(
    _tmp_final_meta,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_meta,
        f,
        indent=2,
    )
os.replace(
    _tmp_final_meta,
    final_meta_path,
)

# Clean up stray temporary checkpoint files if a previous interrupted
# execution left one behind. Valid .pt files are untouched.
for filename in os.listdir(RESUME_OUTPUT_DIR):
    if filename.endswith(".pt.tmp") or filename.endswith(".json.tmp"):
        try:
            os.remove(
                os.path.join(
                    RESUME_OUTPUT_DIR,
                    filename,
                )
            )
        except OSError:
            pass


# ================================================================
# 37. FINAL REPORT
# ================================================================

elapsed = time.time() - start_time
trained_steps = final_step - start_step
trained_tokens = trained_steps * tokens_per_step
average_tok_sec = (
    trained_tokens / max(elapsed, 1e-6)
)

print("\n" + "=" * 70)
print("RESUMED TRAINING COMPLETE")
print("=" * 70)
print(f"Parameters: {total_params:,}")
print(f"Resume start step: {start_step}")
print(f"Final step: {final_step}")
print(f"Additional tokens trained: {trained_tokens:,}")
print(f"Best validation: {best_val:.4f}")
print(f"Additional training time: {elapsed / 3600:.2f} hours")
print(f"Average throughput: {average_tok_sec:,.0f} tok/s")
print("\nArtifacts:")
print(f"  BEST model : {os.path.join(RESUME_OUTPUT_DIR, 'best_model.pt')}")
print(f"  LAST model : {os.path.join(RESUME_OUTPUT_DIR, 'last_model.pt')}")
print(f"  LAST optim : {os.path.join(RESUME_OUTPUT_DIR, 'last_optimizer.pt')}")
print(f"  FINAL model: {final_model_path}")
print(f"  Metadata   : {final_meta_path}")
print("\nNo checkpoint_XXXXX.pt files were created.")

memory_report()


Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB

Checking files...
acts_train           True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
sc_train             True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
tribunals_train      True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
acts_val             True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
sc_val               True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
tribunals_val        True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
tokenizer            True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
test                 True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json

Loading tokenizer...
Tokenizer loaded.

Training data:
acts        : 145,554,561 tokens
sc          : 249,782,556 tokens
tribunals   

W1006 03:39:26.665000 22 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


torch.compile: ENABLED

ARCHITECTURE 4 — RESUME CONFIG
Parameters: 195,151,872
Parameters (M): 195.152
Resume step: 4000
Target step: 10000
Remaining steps: 6000
Remaining tokens: 98,304,000
Batch: 4
Gradient accumulation: 4
Sequence length: 1024
Tokens / microbatch: 4,096
Tokens / optimizer step: 16,384
Fused AdamW: True
torch.compile: True
Best validation so far: 2.8590
Output directory: /kaggle/working/arch4_hybrid_gqa_195m_resume
NEW BEST SAVED:
  /kaggle/working/arch4_hybrid_gqa_195m_resume/best_model.pt
  /kaggle/working/arch4_hybrid_gqa_195m_resume/best_meta.json

TRAINING SPEED WARMUP
Warmup throughput: 6,993 tok/s
GPU memory | allocated=2.23 GB | reserved=9.84 GB | peak=9.47 GB

STARTING RESUMED ARCHITECTURE 4 TRAINING
Starting from step 4000; continuing through step 10000.
STEP  4001/10000 | loss 2.8523 | lr 2.192e-04 | grad 0.656 | 9,068 tok/s
STEP  4050/10000 | loss 3.0256 | lr 2.172e-04 | grad 0.965 | 8,812 tok/s
STEP  4100/10000 | loss 2.9516 | lr 2.152e-04 | grad 0.689 |